In [4]:
!mkdir /kaggle/working/LRNet/demo/landmarks

In [ ]:
!git clone https://github.com/frederickszk/LRNet.git

In [2]:
%cd /kaggle/working/LRNet

/kaggle/working/LRNet


In [ ]:
!pip install -r requirements.txt

In [11]:
text ="""import argparse
import os
from tqdm import tqdm, trange
from torch import optim
from os.path import join
from utils.model import *
from utils.logger import Logger
from utils.metric import *
from utils.dataset import Dataset
from configs.loader import load_yaml


def train_loop(model, train_iter, val_iter, optimizer, loss, epochs, device, add_weights_file):
    log_training_loss = []
    log_training_accuracy = []
    log_val_accuracy = []
    best_val_acc = 0.0

    model.to(device)
    model.train()
    for epoch in trange(1, epochs + 1):
        loss_sum, acc_sum, samples_sum = 0.0, 0.0, 0
        for X, y in train_iter:
            # Load data to GPU/cpu
            X = X.to(device)
            y = y.to(device)
            samples_num = X.shape[0]

            # BP
            output = model(X)
            log_softmax_output = torch.log(output)
            l = loss(log_softmax_output, y)
            optimizer.zero_grad()
            l.backward()
            optimizer.step()

            # Generating Log
            loss_sum += l.item() * samples_num
            acc_sum += calculate_accuracy(output, y) * samples_num
            samples_sum += samples_num
        train_acc = acc_sum/samples_sum

        val_acc = evaluate(model, val_iter, device)

        if val_acc >= best_val_acc:
            save_hint = "save the model to {}".format(add_weights_file)
            torch.save(model.state_dict(), add_weights_file)
            best_val_acc = val_acc
        else:
            save_hint = ""

        tqdm.write("epoch:{}, loss:{:.4}, train_acc:{:.4}, test_acc:{:.4}, best_record:{:.4}  "
                   .format(epoch, loss_sum / samples_sum, train_acc, val_acc, best_val_acc)
                   + save_hint)

        log_training_loss.append(loss_sum/samples_sum)
        log_training_accuracy.append(train_acc)
        log_val_accuracy.append(val_acc)

    log = {"loss": log_training_loss, "acc_train": log_training_accuracy, "acc_val": log_val_accuracy}
    return log


def main(args):
    if_gpu = args.gpu
    dataset_name = args.dataset
    dataset_level = args.level
    branch_selection = args.branch
    args_model = load_yaml("configs/args_model.yaml")
    args_train = load_yaml("configs/args_train.yaml")

    BLOCK_SIZE = args_train["BLOCK_SIZE"]
    BATCH_SIZE = args_train["BATCH_SIZE"]

    add_weights = args_train["add_weights"]
    if not os.path.exists(add_weights):
        os.makedirs(add_weights)

    EPOCHS_g1 = args_train["EPOCHS_g1"]
    LEARNING_RATE_g1 = args_train["LEARNING_RATE_g1"]
    weights_name_g1 = args_train["weights_name_g1"]

    EPOCHS_g2 = args_train["EPOCHS_g2"]
    LEARNING_RATE_g2 = args_train["LEARNING_RATE_g2"]
    weights_name_g2 = args_train["weights_name_g2"]

    if if_gpu:
        # Optional to uncomment if some bugs occur.
        # os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        device = 'cpu'

    dataset = Dataset(add_root=args_train["add_dataset_root"],
                      name=dataset_name, level=dataset_level)

    logger = Logger()
    logger.register_status(dataset=dataset,
                           device=device,
                           branch_selection=branch_selection)
    logger.register_args(**args_train, **args_model)
    logger.print_logs_training()

    train_iter_A = None
    train_iter_B = None
    val_iter_A = None
    val_iter_B = None
    if branch_selection == 'g1':
        train_iter_A = dataset.load_data_train_g1(BLOCK_SIZE, BATCH_SIZE)
        val_iter_A = dataset.load_data_val_g1(BLOCK_SIZE, BATCH_SIZE)
    elif branch_selection == 'g2':
        train_iter_B = dataset.load_data_train_g2(BLOCK_SIZE, BATCH_SIZE)
        val_iter_B = dataset.load_data_val_g2(BLOCK_SIZE, BATCH_SIZE)
    elif branch_selection == 'all':
        train_iter_A, train_iter_B = dataset.load_data_train_all(BLOCK_SIZE, BATCH_SIZE)
        val_iter_A, val_iter_B = dataset.load_data_val_all(BLOCK_SIZE, BATCH_SIZE)
    else:
        print("Unknown branch selection:", branch_selection, '. Please check and restart')
        return

    if branch_selection == 'g1' or branch_selection == 'all':
        assert train_iter_A, val_iter_A is not None
        g1 = LRNet(pretrained_path='./weights/torch/g1.pth', freeze_backbone=True)
        params_to_optimize_g1 = filter(lambda p: p.requires_grad, g1.parameters())
        optimizer = optim.Adam(params_to_optimize_g1, lr=LEARNING_RATE_g1)
        loss = nn.NLLLoss()
        add_weights_file = join(add_weights, weights_name_g1)
        log_g1 = train_loop(g1, train_iter_A, val_iter_A, optimizer, loss, EPOCHS_g1, device, add_weights_file)

    if branch_selection == 'g2' or branch_selection == 'all':
        assert train_iter_B, val_iter_B is not None
        g2 = LRNet(pretrained_path='./weights/torch/g2.pth', freeze_backbone=True)
        params_to_optimize_g2 = filter(lambda p: p.requires_grad, g2.parameters())
        optimizer = optim.Adam(params_to_optimize_g2, lr=LEARNING_RATE_g2)
        loss = nn.NLLLoss()
        add_weights_file = join(add_weights, weights_name_g2)
        log_g2 = train_loop(g2, train_iter_B, val_iter_B, optimizer, loss, EPOCHS_g2, device, add_weights_file)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description='Training codes of LRNet (PyTorch version).',
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    parser.add_argument('-g', '--gpu', action='store_true',
                        help="If use the GPU(CUDA) for training."
                        )
    parser.add_argument('-d', '--dataset', type=str,
                        choices=['DF', 'F2F', 'FS', 'NT', 'FF_all'],
                        default='DF',
                        help="Select the dataset used for training. "
                             "Valid selections: ['DF', 'F2F', 'FS', 'NT', 'FF_all'] "
                        )
    parser.add_argument('-l', '--level', type=str,
                        choices=['raw', 'c23', 'c40'],
                        default='raw',
                        help="Select the dataset compression level. "
                             "Valid selections: ['raw', 'c23', 'c40'] ")
    parser.add_argument('-b', '--branch', type=str,
                        choices=['g1', 'g2', 'all'],
                        default='all',
                        help="Select which branch of the LRNet to be trained. "
                             "Valid selections: ['g1', 'g2', 'all']")
    args = parser.parse_args()
    main(args)"""

with open("//kaggle/working/LRNet/training/train.py", "w") as file:
    file.write(text)


In [44]:
text = """BLOCK_SIZE: 30
BATCH_SIZE: 512

add_dataset_root: '/kaggle/input/datasets/mariaspasyuk/landmarks' 

add_weights: './weights/torch'

EPOCHS_g1: 25
LEARNING_RATE_g1: 0.00002
weights_name_g1: 'g1_test.pth'

EPOCHS_g2: 25
LEARNING_RATE_g2: 0.00002
weights_name_g2: 'g2_test.pth'"""
with open("/kaggle/working/LRNet/training/configs/args_train.yaml", "w") as file:
    file.write(text)


In [53]:
text = """BLOCK_SIZE: 30

landmark_path: '/kaggle/input/datasets/mariaspasyuk/landmarks/Origin/raw/val'  

add_weights: './model_weights'
weights_name_g1: 'g1_test.pth'
weights_name_g2: 'g2_test.pth'"""
with open("/kaggle/working/LRNet/demo/configs/args_inference.yaml", "w") as file:
    file.write(text)


In [49]:
!cp /kaggle/working/LRNet/training/weights/torch/g2_test.pth /kaggle/working/LRNet/demo/model_weights/g2_test.pth
!cp /kaggle/working/LRNet/training/weights/torch/g1_test.pth /kaggle/working/LRNet/demo/model_weights/g1_test.pth

In [45]:
%cd /kaggle/working/LRNet/training

/kaggle/working/LRNet/training


In [ ]:
!python /kaggle/working/LRNet/training/train.py

In [47]:
%cd ../demo

/kaggle/working/LRNet/demo


In [ ]:
!python /kaggle/working/LRNet/demo/classify.py

In [20]:
!mkdir /kaggle/working/LRNet/demo/landmarks/huawei

In [14]:
%cd ./training

/kaggle/working/LRNet/training


In [15]:
predictions_text = """b2659357f2ee8313f83821a20db1b0813226482c68f5e19131dce97db234d8ec.txt-Prediction label: Fake; Scores:1.0
c36133b35229c25082c63a6e108d8d363ca56ca1dfc3e5220f2a45c2d2cd4751.txt-Prediction label: Fake; Scores:1.0
5f3b319080be40e48741e1cbee55ff830909981621cfd534bb045dcfe8845808.txt-Prediction label: Fake; Scores:1.0
29fc25fbb151f230afb48c4322d0ec9ec4f3ee33edbb8edcbb58b6df28c44abb.txt-Prediction label: Fake; Scores:0.8888888888888888
0650898bbf97d4392ef2e69797d07c112283ede68fe4d61988429d2d632e2295.txt-Prediction label: Fake; Scores:1.0
41435ad086c9e613caeff15cdf9e311df23efd03d99695314ed041c3bb2a5bcb.txt-Prediction label: Fake; Scores:1.0
d78b121e5529bf2720638a688824b4503f5a61ffc23e2eba3f33fac7dbde72c6.txt-Prediction label: Fake; Scores:0.7777777777777778
75078ff1217a174060d2d6146f156d5026c62420f4cda57070ddb1bac9bae08a.txt-Prediction label: Fake; Scores:1.0
29a15052dbda40f4ff16782b064bc8fa68e6564464b9eff422c07b9d01d63762.txt-Prediction label: Fake; Scores:0.5
583bd303c43672d076c35c9c45612122c267a90c665ba7083558d75f7e298c8a.txt-Prediction label: Fake; Scores:0.9090909090909091
14a20048f0957a65a970cf4f72416985eb94fc26de337a2893a465eb64f10d60.txt-Prediction label: Fake; Scores:1.0
5a7a75e0aed671b3a9254262d47df76bc80cb63b6ba7f031c27c530f6306878e.txt-Prediction label: Fake; Scores:1.0
c445d3577196271a6a6e95fb640895a20069d3253736dce62de25144f1e94e23.txt-Prediction label: Fake; Scores:1.0
0b8879489d462ce5048e93e7e1c0d03b5703d0db080276d0ee944a2d715fc486.txt-Prediction label: Fake; Scores:1.0
7f44014685f045f523017b51013d26eb3168494649a6f6c3381907c4091d6f56.txt-Prediction label: Fake; Scores:1.0
814030248abdceb343e239bc19749e81630597950c0ee011b7ba1d5a96fd85ca.txt-Prediction label: Fake; Scores:0.8888888888888888
27e55f1ab1b92c19dd02130f64716ebce1845da3cd9d395cb5f2d43a35ecc018.txt-Prediction label: Fake; Scores:1.0
42a71a1821d54661e53e033a1d616e7a478027aac52c90979dc117df1b46745c.txt-Prediction label: Fake; Scores:0.5
07e6cf52f48bd103e850a68462b2d2f7d025418f3e250d7ea07dc7eac3e2fb72.txt-Prediction label: Fake; Scores:1.0
03957815a78dc315902d8228968d074aa115cf967f59a109a8c423c2ef7e5151.txt-Prediction label: Fake; Scores:1.0
2d264f8f80bc13165250728211b9604dfa000fd9c12bc428c58bdd63c60231ed.txt-Prediction label: Fake; Scores:0.9090909090909091
9ebd54c84222555b7cf24279cf0a88270d9a2346acfa045d12690ca58f17fd3a.txt-Prediction label: Fake; Scores:0.8888888888888888
854eea185603a2d46b90daea2b598fe715855aea83d5e2c3a3345cac8b71dc16.txt-Prediction label: Fake; Scores:0.5
6d340a46ddc604d036271c6dcdb990d2c5f5dc11888bb69fdf5a32c244fd91f2.txt-Prediction label: Fake; Scores:0.5
4a65b10e0b31f7c1dee8ba3eed3ee1cdf6b5b4655c45ec32dd7df9b4c6ad5821.txt-Prediction label: Fake; Scores:0.9090909090909091
a914aea93739f9d1e8b5edba8875c09de133cf113d275694b1e746faff08cdd8.txt-Prediction label: Fake; Scores:1.0
6c89d1748985c3655ca0ac18514396cfaeb8e6d2937183746386408744892da2.txt-Prediction label: Fake; Scores:1.0
5091be329de9d399650fbd06d01dba071a10554e899dfa724e4a774c0715cf9a.txt-Prediction label: Fake; Scores:1.0
99cdf1dc1b4a6216d50cd84fdd2e4fd677f40a3059dc942f3b6a2393639ead6d.txt-Prediction label: Fake; Scores:0.5
238b4cbb4604910eec804e8dfef3cb526aa1a18fddb888aaaa2e19b17b51f6ee.txt-Prediction label: Fake; Scores:1.0
2f9c9ed4b7b469aa2741fe59f9cd443459d09d36b639c1acd7119e34cbab8d1d.txt-Prediction label: Fake; Scores:1.0
56787376eba611062dbc881aa6db0c42b5120e8b27560111e0d29f9dd2938569.txt-Prediction label: Fake; Scores:1.0
123236c82c8d03ee5cb07d9156124f405ec86fd45cb2d16919642867296e4ab8.txt-Prediction label: Real; Scores:0.4444444444444444
02cf677dff33c05b6574ac6499c5014c24282bef66381fd06b1979bcc61b060b.txt-Prediction label: Fake; Scores:0.6666666666666666
0d6cb35ec5b367d8b31c581e6a1af65d8c4a433103c5709aa6a19178d2d5b91f.txt-Prediction label: Fake; Scores:1.0
079cab4e845c69779d60f9b535e3b8618022d41dc24a4363ef26a506fbd28d92.txt-Prediction label: Fake; Scores:1.0
c488de52cd037e4bdbd8458b6a373df202e7b5bf7e1bc05dd66fb7c998a2c62a.txt-Prediction label: Fake; Scores:0.6666666666666666
10daa5c75e748aad4972443b809e2900567bef9139669406988396d4cfc0550e.txt-Prediction label: Fake; Scores:0.9090909090909091
b209b672c1f4b4d4550f34a80c59c426afd74bb4bba8e384e346f65cfc75cd58.txt-Prediction label: Fake; Scores:1.0
16aca2fb8f9f5ec2d72d58aa8e43631f23a7c96856a7d22fbf45e3ad757fe880.txt-Prediction label: Fake; Scores:1.0
d76d592ba99af397e65a5c9422f59a7924cd301bc9bd6371b81852f0d421701b.txt-Prediction label: Fake; Scores:0.875
337f22fb41d1cc31c9dcc8659c9c23c6df7523d6dbfb74aad9a0821bd90a7057.txt-Prediction label: Real; Scores:0.0
255ecc3a65c87620d5b41c974a03a2fa40fe4733d2faa8db90cfd6e279af0d72.txt-Prediction label: Real; Scores:0.3333333333333333
0ad429c8fdcce50ad448f958b0ea2eb168ab8fbba5a13fa4e705a49f17aac4bc.txt-Prediction label: Fake; Scores:1.0
883e3847e648903c7e4e696a490f91e41bfbf85ec54f704a757b07e160e14102.txt-Prediction label: Fake; Scores:1.0
56ec99b03b6832f67ffea8a3d3b933d2bb00973b3c1d3aeb28585efd497fb2d1.txt-Prediction label: Real; Scores:0.4444444444444444
1eec96b3f7d232bee4df45bacf408d577d9c805fd086847fdb7e0220db85ddf4.txt-Prediction label: Fake; Scores:1.0"""

In [ ]:
!python /kaggle/working/LRNet/demo/extract_landmarks.py -i  /kaggle/input/datasets/mariaspasyuk/huawei-test -o /kaggle/working/LRNet/demo/landmarks/huawei

In [16]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def parse_predictions(predictions_text: str):
    lines = predictions_text.strip().split('\n')
    result = {}
    for line in lines:
        if '-Prediction label: ' not in line:
            continue
        file_part, rest = line.split('.txt-Prediction label: ', 1)
        filename = file_part.strip()
        label_part, scores_part = rest.split('; Scores:')
        pred_label = label_part.strip()
        prob_fake = float(scores_part.strip())
        result[filename] = (pred_label, prob_fake)
    return result

def compute_metrics(predictions_text: str, csv_path: str):
    df_true = pd.read_csv(csv_path)
    pred_dict = parse_predictions(predictions_text)

    data = []
    for _, row in df_true.iterrows():
        fname = row['obj_id']
        true_label = row['label']
        if fname not in pred_dict.keys():
            print(f"Предупреждение: {fname} отсутствует, пропускаем")
            continue
        pred_label, prob_fake = pred_dict[fname]
        data.append({
            'true_label': true_label,
            'pred_label': pred_label,
            'prob_fake': prob_fake
        })

    df = pd.DataFrame(data)
    if df.empty:
        raise ValueError("Нет данных для расчёта метрик")

    label_map = {'Fake': 1, 'Real': 0}
    y_true = df['true_label']
    y_pred = df['pred_label'].map(label_map)
    y_proba = df['prob_fake']

    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    if len(set(y_true)) < 2:
        auc = float('nan')
    else:
        auc = roc_auc_score(y_true, y_proba)

    return {'accuracy': accuracy, 'f1_score': f1, 'auc': auc}

metrics = compute_metrics(predictions_text, '/kaggle/input/datasets/mariaspasyuk/huawei-test/d83a0ce6-dc87-46a6-9679-98a71cf91886.csv')
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"F1-score: {metrics['f1_score']:.4f}")
print(f"AUC: {metrics['auc']:.4f}" if not pd.isna(metrics['auc']) else "AUC не определён")

Предупреждение: f42d0eb3eba2a73b3b1585a5727455e5bdfea0d4e6bacfadd4d53705427cf9ae отсутствует, пропускаем
Предупреждение: f15bb35be0002193b3b388b8f83eaae9af2293d65fa002367f953be25fe501a1 отсутствует, пропускаем
Предупреждение: f23db699882c3284c253712736a0b62eeee0cbb19efe40184b774bcec52e4739 отсутствует, пропускаем
Предупреждение: f68c05854a65ff59a3b345cea81479595116c7988a3a8db083a8844010de7ac1 отсутствует, пропускаем
Предупреждение: fe2308bd608c60fbf3b6ccc4f90bdec6c7cff7f438ed34c9469904fc003d37d6 отсутствует, пропускаем
Предупреждение: f4f607123be7534e8f5b502230969c769dd6e4f0d0a1a287855e38f3a7bc0ad0 отсутствует, пропускаем
Accuracy: 0.7447
F1-score: 0.8421
AUC: 0.7922


In [1]:
!zip -r huawei.zip /kaggle/working/LRNet/demo/landmarks/huawei

  adding: kaggle/working/LRNet/demo/landmarks/huawei/ (stored 0%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/5091be329de9d399650fbd06d01dba071a10554e899dfa724e4a774c0715cf9a.txt (deflated 80%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/0650898bbf97d4392ef2e69797d07c112283ede68fe4d61988429d2d632e2295.txt (deflated 81%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/2f9c9ed4b7b469aa2741fe59f9cd443459d09d36b639c1acd7119e34cbab8d1d.txt (deflated 83%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/123236c82c8d03ee5cb07d9156124f405ec86fd45cb2d16919642867296e4ab8.txt (deflated 83%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/b2659357f2ee8313f83821a20db1b0813226482c68f5e19131dce97db234d8ec.txt (deflated 85%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/b209b672c1f4b4d4550f34a80c59c426afd74bb4bba8e384e346f65cfc75cd58.txt (deflated 81%)
  adding: kaggle/working/LRNet/demo/landmarks/huawei/42a71a1821d54661e53e033a1d616e7a478027aac52c90979dc117df1b4